# 🗄️ LangChain Tools — Long-Term Memory (Store) & Stream Writer
### Persistent Storage across conversations using InMemoryStore + Real-time Streaming
> **Python 3.11 | LangChain v1.0.0 | .env config**


## Cell 1 — Install & Imports

In [ ]:
%pip install langchain-openai python-dotenv langgraph -q

from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
import os
from dotenv import load_dotenv

load_dotenv()
print("✅ Ready!")

## Cell 2 — LLM Setup

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("API_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0,
)
print("✅ LLM configured!")

## Cell 3 — What is Store?

| Memory Type | Where stored      | Survives conversation? | Access via        |
|-------------|-------------------|------------------------|-------------------|
| State       | In-memory (graph) | ❌ No — resets         | runtime.state     |
| Context     | Passed at invoke  | ❌ No — per invoke     | runtime.context   |
| Store       | Persistent store  | ✅ Yes — always        | runtime.store     |

### Store uses namespace + key pattern:
```python
store.put((namespace,), key, value)   # save
store.get((namespace,), key)          # read
store.delete((namespace,), key)       # delete
store.search((namespace,))            # list all
```
> For production: use `PostgresStore` instead of `InMemoryStore`


## Cell 4 — Define Store Tools (Save & Get & Delete)

In [ ]:
# SAVE to store
@tool
def save_user_info(
    user_id   : str,
    user_info : dict[str, Any],
    runtime   : ToolRuntime,
) -> str:
    """Save user info to long-term memory store."""
    store = runtime.store
    store.put(("users",), user_id, user_info)    # namespace=("users",), key=user_id
    print(f"   💾 Saved user '{user_id}' to store")
    return f"Successfully saved user info for '{user_id}'."


# GET from store
@tool
def get_user_info(
    user_id : str,
    runtime : ToolRuntime,
) -> str:
    """Look up user info from long-term memory store."""
    store     = runtime.store
    user_info = store.get(("users",), user_id)   # namespace=("users",), key=user_id
    if user_info:
        print(f"   📖 Found user '{user_id}' in store")
        return str(user_info.value)
    print(f"   ❌ User '{user_id}' not found in store")
    return f"No info found for user '{user_id}'."


# DELETE from store
@tool
def delete_user_info(
    user_id : str,
    runtime : ToolRuntime,
) -> str:
    """Delete a user's info from the store."""
    store = runtime.store
    store.delete(("users",), user_id)
    print(f"   🗑️  Deleted user '{user_id}' from store")
    return f"Deleted user '{user_id}' from memory."


print("✅ Store tools defined!")

## Cell 5 — Create Store & Agent

In [ ]:
# InMemoryStore — for dev/demo
# For production use: PostgresStore, RedisStore etc.

store = InMemoryStore()

agent = create_agent(
    llm,
    tools=[save_user_info, get_user_info, delete_user_info],
    store=store,                          # attach store to agent
    system_prompt=(
        "You are a helpful assistant with long-term memory. "
        "Use tools to save and retrieve user information from the store."
    )
)

print("✅ Agent created with InMemoryStore!")

## Cell 6 — Session 1: SAVE User Info

In [ ]:
# Simulating first conversation — user shares their info

print("=" * 55)
print("📅 SESSION 1 — Saving user info")
print("=" * 55)

result = agent.invoke({
    "messages": [HumanMessage(content=(
        "Save the following user: "
        "userid: abc123, name: Alice Johnson, age: 30, email: alice@example.com"
    ))]
})

print("\n🤖", result["messages"][-1].content)

## Cell 7 — Session 2: GET User Info (New Conversation)

In [ ]:
# Simulating a BRAND NEW conversation — state is reset
# But store persists → user info is still there!

print("=" * 55)
print("📅 SESSION 2 — New conversation, retrieving user info")
print("=" * 55)

result = agent.invoke({
    "messages": [HumanMessage(content=(
        "Get user info for user with id 'abc123'"
    ))]
})

print("\n🤖", result["messages"][-1].content)

## Cell 8 — Save Multiple Users

In [ ]:
print("=" * 55)
print("📅 Saving multiple users")
print("=" * 55)

result = agent.invoke({
    "messages": [HumanMessage(content=(
        "Save this user: userid: bob456, name: Bob Smith, age: 25, email: bob@example.com"
    ))]
})
print("🤖", result["messages"][-1].content)

result = agent.invoke({
    "messages": [HumanMessage(content=(
        "Save this user: userid: priya789, name: Priya Sharma, age: 28, email: priya@example.com"
    ))]
})
print("🤖", result["messages"][-1].content)

## Cell 9 — Directly Inspect the Store

In [ ]:
# Access the store directly (outside of agent)
# to verify what's saved

print("📦 Store contents (direct access):\n")

all_users = store.search(("users",))   # list all in namespace

for item in all_users:
    print(f"   Key   : {item.key}")
    print(f"   Value : {item.value}")
    print()

## Cell 10 — DELETE from Store

In [ ]:
print("=" * 55)
print("🗑️  Deleting a user from store")
print("=" * 55)

result = agent.invoke({
    "messages": [HumanMessage(content="Delete user info for userid: bob456")]
})
print("🤖", result["messages"][-1].content)

# Verify deletion
print("\n📦 Store after deletion:")
for item in store.search(("users",)):
    print(f"   {item.key} → {item.value}")

## ✅ Summary

### Store (Long-Term Memory)
```python
store.put((namespace,), key, value)   # save data
store.get((namespace,), key)          # read data  → item.value
store.delete((namespace,), key)       # delete data
store.search((namespace,))            # list all   → [item.key, item.value]
```

### Attach to Agent
```python
store = InMemoryStore()
agent = create_agent(model, tools, store=store)
```

### In Tool
```python
store = runtime.store
store.put(("users",), user_id, {"name": "Alice"})
item  = store.get(("users",), user_id)
value = item.value
```